# Original Dataset Usage

In [ ]:
from emma import EmmaDataset, EmmaSearcher, SearchQuery

In [ ]:
dataset_root = None
ds = EmmaDataset(None)

In [ ]:
print(len(ds))
print(ds.root)
print(ds.split_names)
print(ds.num_materials)
print(ds.num_custom_attributes)

In [ ]:
asin = ds.asins[0]
print(asin)

rec = ds.get_record(asin=asin)
print(rec)

print(ds.get_materials(rec=rec))

print(ds.get_category_names(rec=rec))

print(ds.get_category_path(rec=rec))

print(ds.get_price(rec=rec))

print(ds.get_mass(rec=rec))

print(ds.get_ratings(rec=rec))

print(ds.get_package_dimensions(rec=rec))

print(ds.get_quantity(rec=rec))

print(ds.get_custom_attributes(rec=rec))

hashes = ds.get_image_hashes(rec=rec)
print(hashes)

print(ds.get_images(rec=rec))
print(ds.get_images(image_hashes=hashes))

print(ds.get_title(rec=rec))

print(ds.get_features(rec=rec))

print(ds.get_description(rec=rec))

print(ds.get_em_flags(rec=rec))

In [ ]:
se = EmmaSearcher(ds)

In [ ]:
query = SearchQuery(splits=["Train"], require_listed=["mass"], require_present=["package_dimensions"], quantity_range=[1,1])
asins = se.query_asins(query)
print(len(asins))

query = SearchQuery(splits=["Test"], require_listed=["mass"], text_query="stainless steel", text_fields=("title", "features"))
asins = se.query_asins(query)
print(len(asins))

query = SearchQuery(splits=["Test"], require_listed=["mass"], text_query=r"\bsteel\b", text_fields=("description",), text_regex=True)
asins = se.query_asins(query)
print(len(asins))

# Augmentation Usage

In [ ]:
# 1. Generate & store point clouds
import numpy as np
from emma import EmmaDataset, AugmentedDataset
from emma.augment import DirectorySource

ds  = EmmaDataset("/data/emma")

# Create a writable source (directory created on first .save())
pcs = DirectorySource("point_cloud", "/data/emma/pointclouds", suffix=".npy")

# Generate a point cloud for each item (or a subset)
for asin in ds.asins[:1000]:
    rec = ds.get_record(asin)
    pc  = some_pipeline(rec)           # → (N, 3) float32
    pcs.save(asin, pc)

pcs.refresh_index()                    # pick up newly written files
print(pcs)  # DirectorySource(name='point_cloud', n_items=1,000)

In [ ]:
# 2. Load & query augmented dataset
from emma import EmmaDataset, AugmentedDataset, EmmaSearcher
from emma.augment import DirectorySource

ds  = EmmaDataset("/data/emma")
pcs = DirectorySource("point_cloud", "/data/emma/pointclouds")
ads = AugmentedDataset(ds).register(pcs)

# Attribute-style access
pc = ads.point_cloud["B01EL5834O"]        # np.ndarray

# Or explicit
pc = ads.get_augmentation("point_cloud", "B01EL5834O")

# Merged record
rec = ads.get_augmented_record("B01EL5834O")
rec["point_cloud"]                         # included automatically

# All EmmaDataset methods work transparently
print(ads.get_price(ads["B01EL5834O"]))
print(ads.get_materials(ads["B01EL5834O"], top_k=3))

# Search engine works with the underlying dataset
searcher = EmmaSearcher(ads.dataset)
asins    = searcher.query_asins(
    splits=["train"], materials=["plastic"], limit=50,
)
# Retrieve point clouds only for items that have them
pcs_batch = ads.point_cloud.get_batch(asins)

In [ ]:
# 3. Multiple augmentations

from emma.augment import DirectorySource, HDF5Source, InMemorySource

ads = AugmentedDataset(ds)
ads.register(DirectorySource("point_cloud",  "/data/emma/pointclouds"))
ads.register(DirectorySource("depth_map",    "/data/emma/depth",  suffix=".npz"))
ads.register(HDF5Source("embeddings",        "/data/emma/clip.h5"))
ads.register(InMemorySource("my_labels",     precomputed_dict))

# All accessible uniformly
print(ads.source_names)  # ['point_cloud', 'depth_map', 'embeddings', 'my_labels']

rec = ads.get_augmented_record(asin, include=["point_cloud", "embeddings"])

In [ ]:
# 4. Custom storage backend

from emma.augment import AugmentSource

class S3Source(AugmentSource):
    """Read augmentation data from an S3 bucket."""

    def __init__(self, name: str, bucket: str, prefix: str = ""):
        import boto3
        self._name   = name
        self._s3     = boto3.client("s3")
        self._bucket = bucket
        self._prefix = prefix
        self._manifest: set[str] = self._load_manifest()

    @property
    def name(self) -> str:
        return self._name

    def __getitem__(self, asin: str) -> np.ndarray:
        ...

    def __contains__(self, asin: str) -> bool:
        return asin in self._manifest

    def __len__(self) -> int:
        return len(self._manifest)

    def keys(self):
        return self._manifest

# Torch DataLoader Usage

In [ ]:
from torch.utils.data import DataLoader
from emma.dataloader import EmmaMapDataset
from emma.dataloader import emma_collate

In [ ]:
val_asins   = se.query_asins(splits=["Val"],   require_listed=["mass"])

In [ ]:
def my_transform(asin: str, rec: dict):
    return {
        "images": ds.get_images(rec),
        "price": ds.get_price(rec),
        "mass":  ds.get_mass(rec),
        "features": ds.get_features(rec),
        "materials": ds.get_material_probs(rec),
    }

In [ ]:
val_loader = DataLoader(
    EmmaMapDataset(ds, val_asins, transform=my_transform),
    batch_size=2, collate_fn=emma_collate,
)


In [ ]:
data = next(iter(val_loader))

In [ ]:
print(data["images"][0])
print(data["price"][0])
print(data["mass"][0])
print(data["features"][0])
print(data["materials"][0])